# Pipeline 1 — Razonamiento, búsqueda y resultados
**Caso:** asignación de unidad de acarreo en minería de cobre. **Ruta:** Problema → Espacio de estados → Exploración → Búsqueda → Resultado. Valores ilustrativos del caso entregado.

**Aclaración:** la puntuación multicriterio del notebook original está etiquetada allí como A*, pero matemáticamente **no es el algoritmo A***; aquí se la presenta como búsqueda informada basada en score para evitar afirmar una garantía que no existe.

## 1. Problema
Seleccionar entre A, B, C y D un frente viable. El camión no puede ser enviado a un yacimiento agotado o inaccesible. Como simplificación operativa se fija un camión de 150 t y carga prevista de 120 t, parámetros supuestos y ajenos al notebook original.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

url = 'https://raw.githubusercontent.com/josue456cris/Mineria-Pipeline/main/data/yacimientos.csv'
local = Path('../data/yacimientos.csv')
# En Colab tras abrir desde GitHub, se descargan datos públicos del mismo repositorio.
try:
    datos = pd.read_csv(local) if local.exists() else pd.read_csv(url)
except Exception:
    datos = pd.DataFrame([
        ['A', 5000, 12, .80, 1], ['B', 12000, 25, .60, 1],
        ['C', 0, 8, .00, 1], ['D', 8000, 15, .90, 1]],
        columns=['yacimiento','reservas_ton','distancia_km','ley_cobre_pct','acceso_vial'])
display(datos)

## 2. Espacio de estados
Un nodo raíz representa el camión libre; cada yacimiento representa una acción sucesora. Se modelan condiciones de factibilidad antes de comparar scores.

In [ ]:
CAPACIDAD_CAMION_TON = 150  # supuesto didáctico
CARGA_PLANIFICADA_TON = 120  # supuesto didáctico
estados = datos.copy()
estados['factible'] = ((estados.reservas_ton >= CARGA_PLANIFICADA_TON) &
                      (estados.acceso_vial == 1) &
                      (CARGA_PLANIFICADA_TON <= CAPACIDAD_CAMION_TON))
display(estados)
import networkx as nx
G = nx.DiGraph()
for f in datos.yacimiento: G.add_edge('Camión disponible', f'Frente {f}')
pos = nx.spring_layout(G, seed=7)
plt.figure(figsize=(9, 5))
nx.draw(G, pos, with_labels=True, node_size=2600, node_color='#dceaf4', font_size=9, arrows=True)
plt.title('Espacio de estados: asignación de unidad')
plt.savefig('espacio_estados.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Exploración
Se enumeran todos los sucesores. El frente C tiene reservas nulas y se descarta.

In [ ]:
for _, r in estados.iterrows():
    print(f"Frente {r.yacimiento}: reservas={r.reservas_ton:.0f} t, distancia={r.distancia_km:.0f} km, válido={r.factible}")
validos = estados[estados.factible].copy()
assert set(validos.yacimiento) == {'A','B','D'}

## 4. Búsqueda
Se comparan tres criterios. La **exhaustiva** elige el mínimo score entre todos los estados válidos; la estrategia **voraz** elige máximo de reservas; la **informada multicriterio** ordena estados según score. Exhaustiva e informada coinciden aquí porque solo hay tres candidatos, no un grafo de rutas.

In [ ]:
validos['score'] = (validos.distancia_km - validos.reservas_ton / 1000
                   - 10 * validos.ley_cobre_pct)
exhaustiva = validos.loc[validos.score.idxmin(), 'yacimiento']
voraz = validos.loc[validos.reservas_ton.idxmax(), 'yacimiento']
informada = validos.sort_values('score').iloc[0].yacimiento
print('Exhaustiva (mejor score):', exhaustiva)
print('Voraz (mayores reservas):', voraz)
print('Informada (menor score):', informada)
display(validos[['yacimiento','reservas_ton','distancia_km','ley_cobre_pct','score']].sort_values('score'))
assert exhaustiva == informada == 'D' and voraz == 'B' 

## 5. Resultados
De acuerdo con el score didáctico, A = -1, B = 7 y D = -2. Por tanto, **D** es el menor valor; la selección solo por reservas favorece **B**. El resultado depende de los pesos y supuestos, no prueba una optimización real de la mina.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(validos.yacimiento, validos.score, color=['#6788a7' if y != informada else '#48a07a' for y in validos.yacimiento])
ax.axhline(0, color='gray', lw=.8)
ax.set(xlabel='Frente', ylabel='Score (menor es preferible)', title='Puntuación multicriterio por frente')
for i, (_, r) in enumerate(validos.iterrows()):
    ax.text(i, r.score + .2, f'{r.score:.1f}', ha='center')
plt.tight_layout()
plt.show()
print('Conclusión: la heurística selecciona', informada, 'y la regla de reservas selecciona', voraz)
